# BQML HR Analytics — 7 Questions

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "project-2df9d2ad-b6f5-4ed5-997"
DATASET = "hr_bqml_lab"

client = bigquery.Client(project=PROJECT_ID)
client.create_dataset(DATASET, exists_ok=True)

def run(sql):
    return client.query(sql).result().to_dataframe()


## Load HR data

In [ ]:
BASE = "https://raw.githubusercontent.com/accordion-gcp/resources/main/DAY-1"

departments_df = pd.read_csv(f"{BASE}/departments.csv")
employees_df = pd.read_csv(f"{BASE}/employees.csv", parse_dates=["date_of_joining"])
employees_df["date_of_joining"] = employees_df["date_of_joining"].dt.date
attendance_df = pd.read_csv(f"{BASE}/attendance_monthly.csv", parse_dates=["month_date"])
attendance_df["month_date"] = attendance_df["month_date"].dt.date

job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(departments_df, f"{PROJECT_ID}.{DATASET}.departments", job_config=job_config).result()
client.load_table_from_dataframe(employees_df, f"{PROJECT_ID}.{DATASET}.employees", job_config=job_config).result()
client.load_table_from_dataframe(attendance_df, f"{PROJECT_ID}.{DATASET}.attendance_monthly", job_config=job_config).result()
print("data loaded")


## Feature view
One row per employee: attributes + aggregated attendance behavior.

In [ ]:
run(f"""
CREATE OR REPLACE VIEW `{PROJECT_ID}.{DATASET}.employee_features_view` AS
SELECT
  e.employee_id, e.gender, e.state, d.department_name, e.designation,
  DATE_DIFF(CURRENT_DATE(), e.date_of_joining, DAY) AS tenure_days,
  e.monthly_salary_inr,
  ROUND(AVG(a.days_present), 2) AS avg_days_present,
  ROUND(AVG(a.leaves_taken), 2) AS avg_leaves_taken,
  ROUND(AVG(a.performance_rating), 2) AS avg_performance_rating,
  (AVG(a.performance_rating) >= 3.7) AS high_performer
FROM `{PROJECT_ID}.{DATASET}.employees` e
JOIN `{PROJECT_ID}.{DATASET}.departments` d ON e.department_id = d.department_id
JOIN `{PROJECT_ID}.{DATASET}.attendance_monthly` a ON e.employee_id = a.employee_id
GROUP BY e.employee_id, e.gender, e.state, d.department_name, e.designation,
         e.date_of_joining, e.monthly_salary_inr
""")

display(run(f"SELECT * FROM `{PROJECT_ID}.{DATASET}.employee_features_view` LIMIT 10"))


## Q1 — Linear Regression
Goal: predict `monthly_salary_inr` from role, department, tenure.

In [ ]:
run(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.salary_model`
OPTIONS(model_type='linear_reg', input_label_cols=['monthly_salary_inr']) AS
SELECT department_name, designation, state, gender, tenure_days, monthly_salary_inr
FROM `{PROJECT_ID}.{DATASET}.employee_features_view`
""")

display(run(f"SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.salary_model`)"))


In [ ]:
display(run(f"""
SELECT
  employee_id, department_name, designation, monthly_salary_inr AS actual_salary,
  ROUND(predicted_monthly_salary_inr, 2) AS predicted_salary,
  ROUND(monthly_salary_inr - predicted_monthly_salary_inr, 2) AS difference
FROM ML.PREDICT(MODEL `{PROJECT_ID}.{DATASET}.salary_model`, (
  SELECT employee_id, department_name, designation, state, gender, tenure_days, monthly_salary_inr
  FROM `{PROJECT_ID}.{DATASET}.employee_features_view`
))
ORDER BY ABS(difference) DESC
LIMIT 10
"""))


## Q2 — Logistic Regression
Goal: classify employees as high performers (yes/no).

In [ ]:
run(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.high_performer_logistic_model`
OPTIONS(model_type='logistic_reg', input_label_cols=['high_performer']) AS
SELECT department_name, designation, state, gender, tenure_days,
       avg_days_present, avg_leaves_taken, high_performer
FROM `{PROJECT_ID}.{DATASET}.employee_features_view`
""")

display(run(f"SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.high_performer_logistic_model`)"))


## Q3 — Boosted Tree
Goal: same task as Q2, stronger algorithm — compare the two.

In [ ]:
run(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.high_performer_boosted_tree_model`
OPTIONS(model_type='boosted_tree_classifier', input_label_cols=['high_performer']) AS
SELECT department_name, designation, state, gender, tenure_days,
       avg_days_present, avg_leaves_taken, high_performer
FROM `{PROJECT_ID}.{DATASET}.employee_features_view`
""")

logistic_eval = run(f"SELECT 'logistic_reg' AS model, * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.high_performer_logistic_model`)")
boosted_eval = run(f"SELECT 'boosted_tree' AS model, * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.high_performer_boosted_tree_model`)")
display(pd.concat([logistic_eval, boosted_eval], ignore_index=True))


## Q4 — Explainability
Goal: see which features drive the predictions, globally and per-employee.

In [ ]:
display(run(f"""
SELECT * FROM ML.FEATURE_IMPORTANCE(MODEL `{PROJECT_ID}.{DATASET}.high_performer_boosted_tree_model`)
ORDER BY importance_gain DESC
"""))


In [ ]:
display(run(f"""
SELECT employee_id, predicted_high_performer, top_feature_attributions
FROM ML.EXPLAIN_PREDICT(MODEL `{PROJECT_ID}.{DATASET}.high_performer_boosted_tree_model`, (
  SELECT employee_id, department_name, designation, state, gender, tenure_days,
         avg_days_present, avg_leaves_taken
  FROM `{PROJECT_ID}.{DATASET}.employee_features_view`
  LIMIT 5
), STRUCT(3 AS top_k_features))
"""))


## Q5 — K-Means
Goal: find natural employee segments from attendance behavior (no label).

In [ ]:
run(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.employee_segments_model`
OPTIONS(model_type='kmeans', num_clusters=3) AS
SELECT avg_days_present, avg_leaves_taken, avg_performance_rating
FROM `{PROJECT_ID}.{DATASET}.employee_features_view`
""")

display(run(f"""
SELECT centroid_id, feature, ROUND(numerical_value, 2) AS value
FROM ML.CENTROIDS(MODEL `{PROJECT_ID}.{DATASET}.employee_segments_model`)
ORDER BY centroid_id, feature
"""))


In [ ]:
display(run(f"""
SELECT
  f.employee_id, f.department_name, f.avg_days_present, f.avg_leaves_taken,
  f.avg_performance_rating, p.centroid_id
FROM ML.PREDICT(MODEL `{PROJECT_ID}.{DATASET}.employee_segments_model`, (
  SELECT employee_id, avg_days_present, avg_leaves_taken, avg_performance_rating
  FROM `{PROJECT_ID}.{DATASET}.employee_features_view`
)) AS p
JOIN `{PROJECT_ID}.{DATASET}.employee_features_view` AS f USING (employee_id)
ORDER BY p.centroid_id
LIMIT 15
"""))


## Q6 — Anomaly Detection
Goal: flag employees whose attendance pattern doesn't fit any segment.

In [ ]:
display(run(f"""
SELECT employee_id, is_anomaly, ROUND(normalized_distance, 3) AS normalized_distance, centroid_id
FROM ML.DETECT_ANOMALIES(
  MODEL `{PROJECT_ID}.{DATASET}.employee_segments_model`,
  STRUCT(0.1 AS contamination),
  (SELECT employee_id, avg_days_present, avg_leaves_taken, avg_performance_rating
   FROM `{PROJECT_ID}.{DATASET}.employee_features_view`)
)
ORDER BY normalized_distance DESC
"""))


## Q7 — Forecasting
Goal: forecast next quarter's company-wide attendance.

In [ ]:
run(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.attendance_forecast_model`
OPTIONS(
  model_type='ARIMA_PLUS',
  time_series_timestamp_col='month_date',
  time_series_data_col='avg_days_present',
  auto_arima=TRUE
) AS
SELECT month_date, AVG(days_present) AS avg_days_present
FROM `{PROJECT_ID}.{DATASET}.attendance_monthly`
GROUP BY month_date
""")

display(run(f"SELECT * FROM ML.ARIMA_EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.attendance_forecast_model`)"))


In [ ]:
display(run(f"""
SELECT
  forecast_timestamp, ROUND(forecast_value, 2) AS forecast_avg_days_present,
  ROUND(prediction_interval_lower_bound, 2) AS lower_bound,
  ROUND(prediction_interval_upper_bound, 2) AS upper_bound
FROM ML.FORECAST(MODEL `{PROJECT_ID}.{DATASET}.attendance_forecast_model`, STRUCT(3 AS horizon, 0.95 AS confidence_level))
ORDER BY forecast_timestamp
"""))


In [ ]:
# Cleanup -- uncomment to drop the whole dataset (all tables, views, models)
# client.delete_dataset(f"{PROJECT_ID}.{DATASET}", delete_contents=True, not_found_ok=True)
